# HW10 Part B :: Object Detection with Python

COSC 6373 -- Adam Nelson-Archer, 2140122

In [ ]:
!pip install -q transformers torch torchvision Pillow matplotlib requests

In [ ]:
import os
import torch
import requests
from PIL import Image, ImageDraw, ImageFont
import matplotlib.pyplot as plt
from transformers import Owlv2Processor, Owlv2ForObjectDetection

### 1. Detect and localize all person-like objects in an image using open-vocabulary prompts.

We'll use the `google/owlv2-base-patch16-ensemble` model from HuggingFace.

In [ ]:
# Load the OWLv2 processor and model
model_name = "google/owlv2-base-patch16-ensemble"
processor = Owlv2Processor.from_pretrained(model_name)
model = Owlv2ForObjectDetection.from_pretrained(model_name)

print("Model and processor loaded successfully.")

### 2. Load an image you find online from a URL.

In [ ]:
# Load an image from a URL
image_url = "https://images.unsplash.com/photo-1517649763962-0c623066013b?q=80&w=2070&auto=format&fit=crop"
req = requests.get(image_url, headers={'User-Agent': 'Mozilla/5.0'}, stream=True)
image = Image.open(req.raw).convert("RGB")

plt.figure(figsize=(10, 6))
plt.imshow(image)
plt.axis("off")
plt.title("Original Image")
plt.show()

### 3. Run OWLv2 with a list of text prompts
Prompts: `["person", "man", "woman", "people", "girl", "boy", "human"]`

### 4. Apply a confidence threshold (e.g., 0.20) to filter detections.
### 5. Keep the number of detections to a fixed value (e.g., 15).

In [ ]:
texts = [["person", "man", "woman", "people", "girl", "boy", "human"]]
inputs = processor(text=texts, images=image, return_tensors="pt")

# Run inference
with torch.no_grad():
    outputs = model(**inputs)

# Target sizes for bounding box un-normalization
target_sizes = torch.tensor([image.size[::-1]])

# Post-process results
confidence_threshold = 0.20
results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=confidence_threshold)[0]

# Filter and extract detections
boxes = results["boxes"].tolist()
scores = results["scores"].tolist()
labels = results["labels"].tolist()

# Pair them up and sort by confidence score (descending)
detections = list(zip(boxes, scores, labels))
detections.sort(key=lambda x: x[1], reverse=True)

# Keep the number of detections to a fixed value (e.g., 15)
max_detections = 15
detections = detections[:max_detections]

print(f"Found {len(detections)} detections above the threshold.")

### 6. Expand each bounding box slightly using a padding ratio.
### 7. Save each detected region as a cropped image to a `person_crops/` folder.
#### a. Draw green bounding boxes and label them on the original image.

In [ ]:
# Configuration
padding_ratio = 0.05
output_folder = "person_crops"
os.makedirs(output_folder, exist_ok=True)

# Make a copy of the image to draw on
image_with_boxes = image.copy()
draw = ImageDraw.Draw(image_with_boxes)

# Optional: try to load a default font
try:
    font = ImageFont.truetype("arial.ttf", 24)
except IOError:
    font = ImageFont.load_default()

for i, (box, score, label_idx) in enumerate(detections):
    xmin, ymin, xmax, ymax = box
    
    # Calculate padding based on width and height
    width = xmax - xmin
    height = ymax - ymin
    pad_x = width * padding_ratio
    pad_y = height * padding_ratio
    
    # Apply padding and clamp to image boundaries
    new_xmin = max(0, xmin - pad_x)
    new_ymin = max(0, ymin - pad_y)
    new_xmax = min(image.width, xmax + pad_x)
    new_ymax = min(image.height, ymax + pad_y)
    
    # Crop and save
    crop_box = (new_xmin, new_ymin, new_xmax, new_ymax)
    cropped_img = image.crop(crop_box)
    crop_filename = os.path.join(output_folder, f"person_crop_{i+1:02d}.jpg")
    cropped_img.save(crop_filename)
    
    # Draw green bounding box and label on the original image
    draw.rectangle(crop_box, outline="green", width=4)
    
    text_label = f"{texts[0][label_idx]} ({score:.2f})"
    draw.text((new_xmin, new_ymin - 25), text_label, fill="green", font=font)

print(f"Cropped images saved to '{output_folder}/'")

# Show the annotated image
plt.figure(figsize=(12, 8))
plt.imshow(image_with_boxes)
plt.axis("off")
plt.title("Image with Detections")
plt.show()

### 8. Deliverables
Displaying the saved person crop images.

In [ ]:
# Show up to 15 cropped images
crop_files = sorted([f for f in os.listdir(output_folder) if f.endswith(".jpg")])

if crop_files:
    num_crops = len(crop_files)
    cols = 5
    rows = (num_crops // cols) + (1 if num_crops % cols != 0 else 0)
    
    plt.figure(figsize=(15, 3 * rows))
    for i, file in enumerate(crop_files):
        img_path = os.path.join(output_folder, file)
        crop_img = Image.open(img_path)
        
        plt.subplot(rows, cols, i + 1)
        plt.imshow(crop_img)
        plt.axis("off")
        plt.title(f"Crop {i+1}")
        
    plt.tight_layout()
    plt.show()
else:
    print("No crops found.")

### 11. Discussion: Effects of changing confidence threshold and padding ratio

**Confidence Threshold (`threshold = 0.20`):**
- A **higher** confidence threshold (e.g., 0.50) will yield fewer, but more accurate detections. It risks missing partial or occluded people.
- A **lower** confidence threshold (e.g., 0.05) will yield many more detections, potentially capturing people in the background, but introduces a much higher risk of false positives (e.g., identifying objects like statues or clothing as people).

**Padding Ratio (`padding_ratio = 0.05`):**
- A **higher** padding ratio captures more context around the detected person. This is often beneficial if the bounding box is slightly too tight or if downstream tasks require seeing what the person is interacting with. However, too much padding might include other adjacent people or overwhelming background noise.
- A **lower** padding ratio (or 0) creates very tight crops. While this isolates the subject well, it risks cutting off limbs, hair, or edges of the subject if the model's bounding box wasn't perfectly aligned.

## Acknowledgment

I used a coding assistant (Cursor) to help scaffold and organize this notebook.